# ⚾ Pitcher Development Analysis
### Spuhler Field — Full Season TrackMan Report
**Data Range:** April 2025 – January 2026 &nbsp;|&nbsp; **71 Pitchers** &nbsp;|&nbsp; **5,153 Pitches**

---
This notebook provides in-depth pitcher development insights across all tracked sessions. Sections cover:
- **Pitch Arsenal & Usage** — what each pitcher throws and how often
- **Velocity Trends** — how velocity evolves game-to-game
- **Spin Rate & Movement Profiles** — pitch quality and shape
- **Command & Zone Analysis** — control, zone%, and heatmaps
- **Fatigue & Workload** — high-effort outings and pitch-by-pitch velocity fade
- **Pitcher Comparisons** — radar charts and leaderboard tables
- **Development Indicators** — trajectory flags and predictive metrics

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
warnings.filterwarnings("ignore")

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Display wide tables nicely
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)
print("✅ Libraries loaded.")

✅ Libraries loaded.


## 2. Load All Datasets

In [2]:
DATASET_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                           "Analytics", "datasets")
# Resolve relative to this notebook
DATASET_DIR = r"C:\Users\sreey\OneDrive\Desktop\BaseBall\Analytics\datasets"

csv_files = sorted(glob.glob(os.path.join(DATASET_DIR, "*.csv")))
print(f"Found {len(csv_files)} CSV files:\n")

dfs = []
for f in csv_files:
    fname = os.path.basename(f)
    try:
        tmp = pd.read_csv(f, low_memory=False)
        tmp["_source_file"] = fname
        dfs.append(tmp)
        print(f"  ✅ {fname}  — {len(tmp):>5} rows")
    except Exception as e:
        print(f"  ❌ {fname}  — SKIPPED ({e})")

df = pd.concat(dfs, ignore_index=True)

# ── Clean & type-cast ──────────────────────────────────────────────────────
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["RelSpeed"] = pd.to_numeric(df["RelSpeed"], errors="coerce")
df["SpinRate"] = pd.to_numeric(df["SpinRate"], errors="coerce")
df["InducedVertBreak"] = pd.to_numeric(df["InducedVertBreak"], errors="coerce")
df["HorzBreak"] = pd.to_numeric(df["HorzBreak"], errors="coerce")
df["VertBreak"] = pd.to_numeric(df["VertBreak"], errors="coerce")
df["Extension"] = pd.to_numeric(df["Extension"], errors="coerce")
df["RelHeight"] = pd.to_numeric(df["RelHeight"], errors="coerce")
df["RelSide"] = pd.to_numeric(df["RelSide"], errors="coerce")
df["PlateLocHeight"] = pd.to_numeric(df["PlateLocHeight"], errors="coerce")
df["PlateLocSide"] = pd.to_numeric(df["PlateLocSide"], errors="coerce")
df["VertApprAngle"] = pd.to_numeric(df["VertApprAngle"], errors="coerce")
df["HorzApprAngle"] = pd.to_numeric(df["HorzApprAngle"], errors="coerce")

# Remove test/placeholder rows
df = df[~df["Pitcher"].isin(["A, a", "Unknown", ""])]
df = df[df["RelSpeed"].between(40, 110, inclusive="both") | df["RelSpeed"].isna()]

# Derived flags
STRIKE_CALLS = ["StrikeCalled", "StrikeSwinging", "FoulBallNotFieldable",
                "FoulBallFieldable", "InPlay"]
df["IsStrike"] = df["PitchCall"].isin(STRIKE_CALLS)
df["IsSwing"]  = df["PitchCall"].isin(["StrikeSwinging","FoulBallNotFieldable",
                                        "FoulBallFieldable","InPlay"])
df["IsWhiff"]  = df["PitchCall"] == "StrikeSwinging"
df["IsInZone"] = df["PlateLocHeight"].between(1.5, 3.5) & df["PlateLocSide"].between(-0.83, 0.83)
df["IsBall"]   = df["PitchCall"].isin(["BallCalled","BallinDirt"])

print(f"\n✅ Combined dataset: {len(df):,} pitches  |  {df['Pitcher'].nunique()} pitchers  |  {df['Date'].nunique()} session dates")

Found 26 CSV files:

  ✅ 20250426-SpuhlerFieldMain-1_unverified.csv  —   295 rows
  ✅ 20250502-SpuhlerFieldMain-1_unverified.csv  —   228 rows
  ✅ 20250503-SpuhlerFieldMain-1_unverified.csv  —   267 rows
  ✅ 20250504-SpuhlerFieldMain-1_unverified.csv  —   259 rows
  ✅ 20250515-SpuhlerFieldMain-1_unverified.csv  —   264 rows
  ✅ 20250516-SpuhlerFieldMain-1_unverified.csv  —   320 rows
  ✅ 20250517-SpuhlerFieldMain-1_unverified.csv  —   280 rows
  ✅ 20250919-SpuhlerFieldMain-Private-4_unverified.csv  —   142 rows
  ✅ 20250920-SpuhlerFieldMain-Private-1_unverified.csv  —    90 rows
  ✅ 20250926-SpuhlerFieldMain-Private-1_unverified.csv  —   226 rows
  ✅ 20250927-SpuhlerFieldMain-Private-2_unverified.csv  —   141 rows
  ✅ 20251001-SpuhlerFieldMain-Private-1_unverified.csv  —    21 rows
  ✅ 20251004-SpuhlerFieldMain-Private-1_unverified.csv  —   287 rows
  ✅ 20251004-SpuhlerFieldMain-Private-2_unverified.csv  —   286 rows
  ✅ 20251010-SpuhlerFieldMain-Private-1_unverified.csv  —     8 rows


## 3. Pitcher Performance Overview

In [21]:
# ── Per-pitcher summary metrics ───────────────────────────────────────────
pitch_totals = df.groupby("Pitcher").agg(
    Pitches        = ("RelSpeed",  "count"),
    Sessions       = ("Date",      "nunique"),
    Avg_Velo       = ("RelSpeed",  "mean"),
    Max_Velo       = ("RelSpeed",  "max"),
    Avg_SpinRate   = ("SpinRate",  "mean"),
    Avg_IVB        = ("InducedVertBreak", "mean"),
    Avg_HorzBreak  = ("HorzBreak", "mean"),
    Avg_Extension  = ("Extension", "mean"),
    Avg_RelHeight  = ("RelHeight", "mean"),
    Avg_RelSide    = ("RelSide",   "mean"),
    Strike_pct     = ("IsStrike",  "mean"),
    Zone_pct       = ("IsInZone",  "mean"),
    Whiff_pct      = ("IsWhiff",   "mean"),
    Ball_pct       = ("IsBall",    "mean"),
).reset_index()

# Strikeout & walk rates per pitcher (per batter faced proxy)
k_bb = df[df["KorBB"].isin(["Strikeout","Walk"])].groupby(["Pitcher","KorBB"]).size().unstack(fill_value=0)
k_bb.columns = [f"N_{c}" for c in k_bb.columns]
pitch_totals = pitch_totals.merge(k_bb, on="Pitcher", how="left").fillna(0)

# Hands
hands = df.groupby("Pitcher")["PitcherThrows"].agg(lambda x: x.mode()[0]).reset_index()
hands.columns = ["Pitcher","Throws"]
pitch_totals = pitch_totals.merge(hands, on="Pitcher", how="left")

# Sort by pitch count
pitch_totals = pitch_totals.sort_values("Pitches", ascending=False).reset_index(drop=True)

# Format for display
display_cols = ["Pitcher","Throws","Pitches","Sessions","Avg_Velo","Max_Velo",
                "Avg_SpinRate","Strike_pct","Zone_pct","Whiff_pct",
                "N_Strikeout","N_Walk"]
rename_map = {"Avg_Velo":"Avg Velo","Max_Velo":"Max Velo","Avg_SpinRate":"Avg Spin",
              "Strike_pct":"Strike%","Zone_pct":"Zone%","Whiff_pct":"Whiff%",
              "N_Strikeout":"K","N_Walk":"BB"}
fmt = pitch_totals[display_cols].rename(columns=rename_map)
fmt["Strike%"] = (fmt["Strike%"] * 100).round(1)
fmt["Zone%"]   = (fmt["Zone%"]   * 100).round(1)
fmt["Whiff%"]  = (fmt["Whiff%"]  * 100).round(1)
fmt["Avg Velo"]= fmt["Avg Velo"].round(1)
fmt["Avg Spin"]= fmt["Avg Spin"].round(0).astype("Int64")
print("=== PITCHER LEADERBOARD (sorted by pitch count) ===")
print(fmt.to_string(index=False))

=== PITCHER LEADERBOARD (sorted by pitch count) ===
             Pitcher Throws  Pitches  Sessions  Avg Velo  Max Velo  Avg Spin  Strike%  Zone%  Whiff%     K    BB
         Drumm, Jake  Right      266         8     83.70     90.46      2035    58.20  38.80    7.80 12.00 10.00
      Okeeffe, Shaun   Left      258         9     79.80     85.30      2205    54.80  38.60   14.30 16.00 11.00
      Thomas, Parker  Right      251         7     83.20     91.43      1996    53.60  34.50    9.50 13.00 11.00
      O'Hara, Connor  Right      228         3     87.30     94.61      2102    67.00  49.10    9.60 12.00  3.00
      Rumberg, Logan  Right      218         6     85.90     94.25      2304    56.00  43.60   10.10 13.00  8.00
       Wrehe, Thomas  Right      189         8     84.50     89.24      1869    53.40  33.00    7.30  5.00  6.00
     Peters, Jackson  Right      180         6     84.00     91.87      2128    51.10  29.40    7.20 11.00  7.00
       Parker, Aiden  Right      165        

In [29]:
# ── Top 20 pitchers by pitch count — interactive bar ─────────────────────
top20 = pitch_totals.head(20).copy()
top20["Label"] = top20["Pitcher"].str.split(",").str[0]  # last name only

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Average Velocity (mph)", "Average Spin Rate (rpm)"])

colors = px.colors.qualitative.Bold

fig.add_trace(go.Bar(
    x=top20["Label"], y=top20["Avg_Velo"].round(1),
    marker_color=colors[:len(top20)],
    text=top20["Avg_Velo"].round(1), textposition="outside",
    name="Avg Velo"), row=1, col=1)

fig.add_trace(go.Bar(
    x=top20["Label"], y=top20["Avg_SpinRate"].round(0),
    marker_color=colors[:len(top20)],
    text=top20["Avg_SpinRate"].round(0), textposition="outside",
    name="Avg Spin"), row=1, col=2)

fig.update_layout(height=480, title_text="Top 20 Pitchers — Velocity & Spin Rate Overview",
                  showlegend=False, plot_bgcolor="#f9f9f9",
                  font=dict(size=11))
fig.update_xaxes(tickangle=-35)
fig.show()

In [5]:
# ── Strike%, Zone%, Whiff% grouped bar ───────────────────────────────────
fig2 = go.Figure()
metrics = [("Strike_pct","Strike%","#2196F3"),
           ("Zone_pct",  "Zone%",  "#4CAF50"),
           ("Whiff_pct", "Whiff%", "#FF5722")]

for col, name, color in metrics:
    fig2.add_trace(go.Bar(
        x=top20["Label"],
        y=(top20[col]*100).round(1),
        name=name,
        marker_color=color,
        text=(top20[col]*100).round(1),
        textposition="outside",
        texttemplate="%{text}%"
    ))

fig2.update_layout(
    barmode="group",
    title="Top 20 Pitchers — Strike%, Zone%, Whiff%",
    yaxis_title="%",
    height=450,
    plot_bgcolor="#f9f9f9",
    legend=dict(orientation="h", y=1.08),
    font=dict(size=11)
)
fig2.update_xaxes(tickangle=-35)
fig2.show()

## 4. Pitch Arsenal Analysis
Each pitcher's pitch mix shown as usage %. Stacked bar gives team-wide view; individual breakdowns follow.

In [6]:
PITCH_COLORS = {
    "Fastball":       "#E53935",
    "Sinker":         "#EF6C00",
    "TwoSeamFastBall":"#FB8C00",
    "Cutter":         "#FDD835",
    "Slider":         "#43A047",
    "Sweeper":        "#00ACC1",
    "Curveball":      "#1E88E5",
    "ChangeUp":       "#8E24AA",
    "Splitter":       "#6D4C41",
    "Other":          "#9E9E9E",
    "Undefined":      "#BDBDBD",
}

# Filter to top 20 pitchers and relevant pitch types
valid_types = [t for t in PITCH_COLORS if t != "Undefined"]
arsenal_df = df[df["Pitcher"].isin(top20["Pitcher"]) &
                df["TaggedPitchType"].isin(valid_types)].copy()

# Pivot: pitch mix by type
mix = (arsenal_df.groupby(["Pitcher","TaggedPitchType"])
                 .size()
                 .reset_index(name="N"))
mix_total = mix.groupby("Pitcher")["N"].transform("sum")
mix["Pct"] = (mix["N"] / mix_total * 100).round(1)

# Sort by total pitch count (same order as top20)
pitcher_order = top20["Pitcher"].tolist()
mix["Pitcher_short"] = mix["Pitcher"].str.split(",").str[0]
order_map = {p: i for i, p in enumerate(pitcher_order)}
mix["_ord"] = mix["Pitcher"].map(order_map)
mix = mix.sort_values("_ord")

fig3 = px.bar(
    mix, x="Pitcher_short", y="Pct", color="TaggedPitchType",
    color_discrete_map=PITCH_COLORS,
    text="Pct",
    title="Pitch Arsenal — Usage % per Pitcher (top 20)",
    labels={"Pct":"Usage %","TaggedPitchType":"Pitch Type","Pitcher_short":"Pitcher"},
    height=500,
    category_orders={"TaggedPitchType": valid_types}
)
fig3.update_traces(texttemplate="%{text:.0f}%", textposition="inside")
fig3.update_layout(barmode="stack", plot_bgcolor="#f9f9f9",
                   xaxis_tickangle=-35, legend_title="Pitch Type",
                   font=dict(size=11))
fig3.show()

In [7]:
# ── Pitch-type stacked bar: velocity by type (team average) ──────────────
velo_by_type = (df[df["TaggedPitchType"].isin(valid_types)]
                .groupby("TaggedPitchType")
                .agg(Avg_Velo=("RelSpeed","mean"),
                     Count=("RelSpeed","count"),
                     Avg_Spin=("SpinRate","mean"),
                     Avg_IVB=("InducedVertBreak","mean"),
                     Avg_HB=("HorzBreak","mean"))
                .reset_index()
                .sort_values("Avg_Velo", ascending=False))

fig4 = go.Figure(go.Bar(
    x=velo_by_type["Avg_Velo"],
    y=velo_by_type["TaggedPitchType"],
    orientation="h",
    marker_color=[PITCH_COLORS[t] for t in velo_by_type["TaggedPitchType"]],
    text=velo_by_type["Avg_Velo"].round(1),
    textposition="outside",
    customdata=velo_by_type[["Count","Avg_Spin","Avg_IVB","Avg_HB"]].values,
    hovertemplate=(
        "<b>%{y}</b><br>Avg Velo: %{x:.1f} mph<br>"
        "Count: %{customdata[0]:,}<br>Avg Spin: %{customdata[1]:.0f} rpm<br>"
        "Avg IVB: %{customdata[2]:.1f}″<br>Avg HorzBreak: %{customdata[3]:.1f}″<extra></extra>"
    )
))
fig4.update_layout(
    title="Team-Wide Avg Velocity by Pitch Type",
    xaxis_title="Avg Velocity (mph)",
    height=400, plot_bgcolor="#f9f9f9",
    font=dict(size=12)
)
fig4.show()
print("\nPitch Type Summary:")
print(velo_by_type.to_string(index=False))


Pitch Type Summary:
TaggedPitchType  Avg_Velo  Count  Avg_Spin  Avg_IVB  Avg_HB
       Fastball     88.23   2166   2136.95    15.49    6.43
TwoSeamFastBall     86.46     20   2080.64    13.50    0.49
          Other     85.92      2   2155.95     9.70   10.55
         Sinker     85.42    488   1986.96     4.48   12.32
         Cutter     83.39    282   2210.59     3.30   -2.16
       Splitter     80.71     61   1308.43     5.37    9.07
       ChangeUp     79.81    469   1740.36     5.59    5.29
         Slider     78.65   1076   2239.88     0.56   -3.76
        Sweeper     78.42    144   2311.21     1.49  -14.51
      Curveball     75.66    240   2289.70    -7.90   -7.01


In [30]:
# ── Strike Rate vs Pitch Arsenal Comparison ──────────────────────────────
# For each pitcher × pitch type: avg strike%, whiff%, zone%, and usage%
# Reveals which pitch types generate the most strikes and whiffs per pitcher.

strike_by_type = (
    df[df["TaggedPitchType"].isin(valid_types) & df["Pitcher"].isin(top20["Pitcher"])]
    .groupby(["Pitcher", "TaggedPitchType"])
    .agg(
        Count      = ("PitchCall",   "count"),
        Strike_pct = ("IsStrike",    "mean"),
        Whiff_pct  = ("IsWhiff",     "mean"),
        Zone_pct   = ("IsInZone",    "mean"),
        CSW_pct    = ("IsCSW",       "mean"),
        Avg_Velo   = ("RelSpeed",    "mean"),
    )
    .reset_index()
)

# Add usage % within pitcher
strike_by_type["Usage_pct"] = (
    strike_by_type["Count"]
    / strike_by_type.groupby("Pitcher")["Count"].transform("sum")
    * 100
).round(1)

strike_by_type["Short"] = strike_by_type["Pitcher"].str.split(",").str[0]
strike_by_type["Strike_pct_lbl"] = (strike_by_type["Strike_pct"] * 100).round(1)
strike_by_type["Whiff_pct_lbl"]  = (strike_by_type["Whiff_pct"]  * 100).round(1)

# ── Chart 1: Strike% by pitch type (bubble = usage, color = pitch type) ──
fig_sa1 = px.scatter(
    strike_by_type,
    x="Usage_pct",
    y="Strike_pct_lbl",
    color="TaggedPitchType",
    color_discrete_map=PITCH_COLORS,
    size="Count",
    size_max=22,
    facet_col="Short",
    facet_col_wrap=5,
    text="TaggedPitchType",
    title="Strike% vs Usage% by Pitch Type — Top 20 Pitchers",
    labels={"Usage_pct": "Usage %", "Strike_pct_lbl": "Strike %",
            "TaggedPitchType": "Pitch Type", "Short": "Pitcher"},
    height=900,
    category_orders={"TaggedPitchType": valid_types},
)
fig_sa1.update_traces(textposition="top center", textfont_size=8)
fig_sa1.update_layout(showlegend=True, plot_bgcolor="#f9f9f9", font=dict(size=9))
for ann in fig_sa1.layout.annotations:
    ann.text = ann.text.split("=")[-1]
fig_sa1.show()

# ── Chart 2: Grouped bar — Strike% per pitch type (team average) ──────────
type_strike = (
    df[df["TaggedPitchType"].isin(valid_types)]
    .groupby("TaggedPitchType")
    .agg(
        Strike_pct = ("IsStrike", "mean"),
        Whiff_pct  = ("IsWhiff",  "mean"),
        CSW_pct    = ("IsCSW",    "mean"),
        Count      = ("PitchCall","count"),
    )
    .reset_index()
    .sort_values("Strike_pct", ascending=False)
)

fig_sa2 = go.Figure()
for col, name, color in [
    ("Strike_pct", "Strike%", "#2196F3"),
    ("Whiff_pct",  "Whiff%",  "#FF5722"),
    ("CSW_pct",    "CSW%",    "#4CAF50"),
]:
    fig_sa2.add_trace(go.Bar(
        x=type_strike["TaggedPitchType"],
        y=(type_strike[col] * 100).round(1),
        name=name,
        marker_color=color,
        text=(type_strike[col] * 100).round(1),
        texttemplate="%{text:.1f}%",
        textposition="outside",
    ))

fig_sa2.update_layout(
    barmode="group",
    title="Team-Wide Strike%, Whiff%, CSW% by Pitch Type",
    yaxis_title="%",
    height=450,
    plot_bgcolor="#f9f9f9",
    legend=dict(orientation="h", y=1.08),
    font=dict(size=11),
)
fig_sa2.show()

# ── Chart 3: Heatmap — each pitcher's Strike% per pitch type ─────────────
pivot = strike_by_type.pivot_table(
    index="Short", columns="TaggedPitchType",
    values="Strike_pct_lbl", aggfunc="mean"
).fillna(0)

fig_sa3 = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale="RdYlGn",
    zmin=40, zmax=80,
    text=pivot.values.round(1),
    texttemplate="%{text:.0f}%",
    colorbar=dict(title="Strike%"),
))
fig_sa3.update_layout(
    title="Strike% Heatmap — Pitcher × Pitch Type",
    xaxis_title="Pitch Type",
    yaxis_title="Pitcher",
    height=max(400, len(pivot) * 28),
    font=dict(size=11),
)
fig_sa3.show()

# ── Summary table ─────────────────────────────────────────────────────────
print("=== PITCH TYPE vs STRIKE RATE (Team Average) ===")
disp = type_strike.copy()
disp["Strike%"] = (disp["Strike_pct"] * 100).round(1)
disp["Whiff%"]  = (disp["Whiff_pct"]  * 100).round(1)
disp["CSW%"]    = (disp["CSW_pct"]    * 100).round(1)
print(disp[["TaggedPitchType","Count","Strike%","Whiff%","CSW%"]].to_string(index=False))

=== PITCH TYPE vs STRIKE RATE (Team Average) ===
TaggedPitchType  Count  Strike%  Whiff%  CSW%
         Cutter    282    64.90   10.60 33.70
       Fastball   2183    60.10    5.60 23.70
       Splitter     61    59.00   13.10 19.70
         Sinker    488    57.60    8.80 24.80
       ChangeUp    471    56.50   14.20 27.60
         Slider   1077    56.20   10.90 30.20
        Sweeper    144    55.60   14.60 35.40
      Curveball    240    52.50    9.20 29.60
TwoSeamFastBall     21    28.60    0.00  9.50
          Other      9    22.20    0.00 11.10


In [32]:
# ── Average Strikes per Pitch Type ───────────────────────────────────────
# Ranks pitch types by: (1) average strike count per pitcher and
# (2) average strike rate — revealing which pitch is the "best" strike-getter.

# Compute from type_strike (already in memory from previous cell)
ranked = type_strike.copy()
ranked["Avg_Strikes_Per_100"] = (ranked["Strike_pct"] * 100).round(2)  # strike rate per 100 pitches

# Average absolute strike count per pitcher for each pitch type
abs_strikes = (
    df[df["TaggedPitchType"].isin(valid_types)]
    .groupby(["Pitcher", "TaggedPitchType"])["IsStrike"]
    .sum()
    .reset_index()
    .groupby("TaggedPitchType")["IsStrike"]
    .mean()
    .reset_index()
    .rename(columns={"IsStrike": "Avg_Strike_Count_Per_Pitcher"})
)
ranked = ranked.merge(abs_strikes, on="TaggedPitchType")
ranked["Avg_Strike_Count_Per_Pitcher"] = ranked["Avg_Strike_Count_Per_Pitcher"].round(1)
ranked = ranked.sort_values("Avg_Strike_Count_Per_Pitcher", ascending=False)

best_type = ranked.iloc[0]["TaggedPitchType"]
best_rate = ranked.iloc[0]["Avg_Strikes_Per_100"]
best_cnt  = ranked.iloc[0]["Avg_Strike_Count_Per_Pitcher"]

print(f"Best strike-getting pitch: {best_type}  |  "
      f"{best_rate:.1f} strikes per 100 pitches  |  "
      f"{best_cnt:.0f} avg strikes per pitcher")

# Bar chart — Avg strikes per pitcher per pitch type (absolute count)
fig_avg1 = go.Figure()
fig_avg1.add_trace(go.Bar(
    x=ranked["TaggedPitchType"],
    y=ranked["Avg_Strike_Count_Per_Pitcher"],
    marker_color=[PITCH_COLORS.get(p, "#888") for p in ranked["TaggedPitchType"]],
    text=ranked["Avg_Strike_Count_Per_Pitcher"],
    texttemplate="%{text:.0f}",
    textposition="outside",
    name="Avg Strikes per Pitcher",
))
fig_avg1.update_layout(
    title=f"Average Strike Count per Pitcher by Pitch Type  ✦ Best: {best_type} ({best_cnt:.0f} strikes avg)",
    xaxis_title="Pitch Type",
    yaxis_title="Avg Strikes per Pitcher",
    height=430,
    plot_bgcolor="#f9f9f9",
    font=dict(size=12),
    showlegend=False,
)
fig_avg1.show()

# Bar chart — Strike rate (per 100 pitches)
ranked_rate = ranked.sort_values("Avg_Strikes_Per_100", ascending=False)
fig_avg2 = go.Figure()
fig_avg2.add_trace(go.Bar(
    x=ranked_rate["TaggedPitchType"],
    y=ranked_rate["Avg_Strikes_Per_100"],
    marker_color=[PITCH_COLORS.get(p, "#888") for p in ranked_rate["TaggedPitchType"]],
    text=ranked_rate["Avg_Strikes_Per_100"],
    texttemplate="%{text:.1f}%",
    textposition="outside",
    name="Strike Rate",
))
fig_avg2.update_layout(
    title="Strike Rate (per 100 pitches) by Pitch Type",
    xaxis_title="Pitch Type",
    yaxis_title="Strike %",
    height=430,
    plot_bgcolor="#f9f9f9",
    font=dict(size=12),
    showlegend=False,
)
fig_avg2.show()

# Summary printout
print("\n=== RANKED: AVERAGE STRIKES PER PITCHER BY PITCH TYPE ===")
print(ranked[["TaggedPitchType", "Count", "Avg_Strike_Count_Per_Pitcher", "Avg_Strikes_Per_100"]]
      .rename(columns={"Avg_Strikes_Per_100": "Strike% (per 100)"})
      .to_string(index=False))

Best strike-getting pitch: Sweeper  |  55.6 strikes per 100 pitches  |  20 avg strikes per pitcher



=== RANKED: AVERAGE STRIKES PER PITCHER BY PITCH TYPE ===
TaggedPitchType  Count  Avg_Strike_Count_Per_Pitcher  Strike% (per 100)
        Sweeper    144                         20.00              55.56
       Fastball   2183                         19.20              60.05
         Sinker    488                         17.60              57.58
         Cutter    282                         12.10              64.89
         Slider   1077                         10.60              56.17
       ChangeUp    471                          5.80              56.48
      Curveball    240                          4.10              52.50
       Splitter     61                          3.30              59.02
TwoSeamFastBall     21                          1.50              28.57
          Other      9                          0.30              22.22


## 5. Velocity Trends Over Time
Game-by-game average fastball velocity per pitcher. Rolling 3-game average shown to smooth noise.

In [8]:
FASTBALL_TYPES = ["Fastball","Sinker","TwoSeamFastBall","Cutter"]

# Game-level fastball velo per pitcher
fb_trend = (df[df["TaggedPitchType"].isin(FASTBALL_TYPES) &
               df["Pitcher"].isin(top20["Pitcher"])]
            .groupby(["Pitcher","Date"])
            .agg(Avg_FB_Velo=("RelSpeed","mean"),
                 Max_FB_Velo=("RelSpeed","max"),
                 Pitches=("RelSpeed","count"))
            .reset_index())

fb_trend = fb_trend.sort_values(["Pitcher","Date"])
fb_trend["Rolling_Avg"] = (fb_trend.groupby("Pitcher")["Avg_FB_Velo"]
                             .transform(lambda x: x.rolling(3, min_periods=1).mean()))

# Plot top 12 pitchers
top12 = pitch_totals.head(12)["Pitcher"].tolist()

fig5 = px.line(
    fb_trend[fb_trend["Pitcher"].isin(top12)],
    x="Date", y="Rolling_Avg",
    color="Pitcher",
    facet_col="Pitcher", facet_col_wrap=4,
    markers=True,
    labels={"Rolling_Avg":"Avg FB Velo (mph)","Date":"Game Date"},
    title="Fastball Velocity Trend — Rolling 3-Session Avg (Top 12 Pitchers)",
    height=700,
    color_discrete_sequence=px.colors.qualitative.Plotly,
)
fig5.update_traces(line_width=2.5, marker_size=5)
fig5.update_yaxes(range=[78, 100])
fig5.update_layout(showlegend=False, plot_bgcolor="#f9f9f9", font=dict(size=10))
fig5.update_xaxes(tickformat="%b %d", tickangle=-45)
fig5.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].split(",")[0]))
fig5.show()

In [9]:
# ── Velocity gain/loss table: first session vs most recent ──────────────
velo_first_last = (fb_trend.groupby("Pitcher")
                   .apply(lambda x: pd.Series({
                       "First_Date":  x.iloc[0]["Date"],
                       "First_Velo":  x.iloc[0]["Avg_FB_Velo"],
                       "Last_Date":   x.iloc[-1]["Date"],
                       "Last_Velo":   x.iloc[-1]["Avg_FB_Velo"],
                       "Sessions":    len(x),
                   }))
                   .reset_index())
velo_first_last["Velo_Change"] = (velo_first_last["Last_Velo"] - velo_first_last["First_Velo"]).round(2)
velo_first_last = velo_first_last.sort_values("Velo_Change", ascending=False)

fig6 = go.Figure(go.Bar(
    x=velo_first_last["Pitcher"].str.split(",").str[0],
    y=velo_first_last["Velo_Change"],
    marker_color=["#2ecc71" if v >= 0 else "#e74c3c" for v in velo_first_last["Velo_Change"]],
    text=velo_first_last["Velo_Change"].apply(lambda v: f"+{v:.1f}" if v >= 0 else f"{v:.1f}"),
    textposition="outside",
))
fig6.update_layout(
    title="FB Velocity Change: First → Most Recent Session",
    yaxis_title="Velocity Change (mph)",
    height=420, plot_bgcolor="#f9f9f9",
    xaxis_tickangle=-35, font=dict(size=11)
)
fig6.add_hline(y=0, line_dash="dot", line_color="gray")
fig6.show()
print(velo_first_last[["Pitcher","First_Date","First_Velo","Last_Date","Last_Velo","Velo_Change"]].to_string(index=False))

         Pitcher First_Date  First_Velo  Last_Date  Last_Velo  Velo_Change
   Wrehe, Thomas 2025-09-20       85.00 2026-01-16      87.42         2.42
    Knox, Connor 2025-05-02       89.65 2025-05-15      91.65         2.01
  O'Hara, Connor 2025-04-26       88.51 2025-05-16      90.19         1.67
   Kaler, Tanner 2025-09-20       83.99 2026-01-16      85.52         1.52
Cassedy, Brandon 2025-05-04       87.40 2025-05-16      88.31         0.92
    Hsu, Brandon 2025-10-14       86.54 2026-01-16      87.34         0.79
   Parker, Aiden 2025-09-20       87.30 2026-01-18      87.82         0.52
   Willis, Tyson 2025-09-27       86.08 2025-10-31      86.51         0.43
 Cardenas, Diego 2025-09-19       89.53 2025-10-18      89.71         0.18
    Book, Colton 2025-05-04       88.37 2025-05-04      88.37         0.00
   Rabayda, Mike 2025-05-15       90.36 2025-05-15      90.36         0.00
    Ertel, Brant 2025-10-10       85.08 2025-11-07      84.80        -0.28
     Drumm, Jake 2025-05-

## 6. Pitch Movement & Spin Rate Analysis
The movement profile scatter (Induced Vertical Break vs Horizontal Break) fingerprints each pitch type.  
Higher IVB = more "rise" on fastballs; armside run appears on the right for RHP.

In [10]:
# ── Team-wide movement profile (all pitchers, all pitch types) ───────────
mvmt = df[df["TaggedPitchType"].isin(valid_types) &
          df["InducedVertBreak"].notna() &
          df["HorzBreak"].notna()].copy()

fig7 = px.scatter(
    mvmt, x="HorzBreak", y="InducedVertBreak",
    color="TaggedPitchType",
    color_discrete_map=PITCH_COLORS,
    symbol="PitcherThrows",
    opacity=0.45,
    hover_data={"Pitcher":True,"RelSpeed":True,"SpinRate":True},
    title="Team Movement Profile — All Pitchers (IVB vs Horizontal Break)",
    labels={"HorzBreak":"Horizontal Break (in)","InducedVertBreak":"Induced Vertical Break (in)",
            "TaggedPitchType":"Pitch Type","PitcherThrows":"Hand"},
    height=550,
    category_orders={"TaggedPitchType": valid_types}
)
fig7.add_hline(y=0, line_dash="dot", line_color="black", line_width=1)
fig7.add_vline(x=0, line_dash="dot", line_color="black", line_width=1)
fig7.update_layout(plot_bgcolor="#f9f9f9", font=dict(size=11))
fig7.show()

In [12]:
# ── Spin Rate by pitch type — violin + box ───────────────────────────────
spin_df = df[df["TaggedPitchType"].isin(valid_types) & df["SpinRate"].notna()].copy()

fig8 = px.violin(
    spin_df, x="TaggedPitchType", y="SpinRate",
    color="TaggedPitchType",
    color_discrete_map=PITCH_COLORS,
    box=True, points=False,
    title="Spin Rate Distribution by Pitch Type (all pitchers)",
    labels={"SpinRate":"Spin Rate (rpm)","TaggedPitchType":"Pitch Type"},
    height=480,
    category_orders={"TaggedPitchType": valid_types}
)
fig8.update_layout(showlegend=False, plot_bgcolor="#f9f9f9",
                   xaxis_tickangle=-25, font=dict(size=11))
fig8.show()

# ── Spin Rate vs Velocity scatter (colored by pitch type) ────────────────
fig9 = px.scatter(
    spin_df, x="RelSpeed", y="SpinRate",
    color="TaggedPitchType",
    color_discrete_map=PITCH_COLORS,
    opacity=0.4,
    trendline="ols",
    trendline_scope="trace",
    title="Spin Rate vs Velocity by Pitch Type",
    labels={"RelSpeed":"Release Speed (mph)","SpinRate":"Spin Rate (rpm)",
            "TaggedPitchType":"Pitch Type"},
    height=500,
    category_orders={"TaggedPitchType": valid_types}
)
fig9.update_layout(plot_bgcolor="#f9f9f9", font=dict(size=11))
fig9.show()

In [13]:
# ── Per-pitcher movement profile for top 9 pitchers ─────────────────────
top9 = pitch_totals.head(9)["Pitcher"].tolist()
mvmt9 = mvmt[mvmt["Pitcher"].isin(top9)].copy()
mvmt9["Short"] = mvmt9["Pitcher"].str.split(",").str[0]

fig10 = px.scatter(
    mvmt9, x="HorzBreak", y="InducedVertBreak",
    color="TaggedPitchType",
    color_discrete_map=PITCH_COLORS,
    facet_col="Short", facet_col_wrap=3,
    opacity=0.6,
    title="Individual Movement Profiles — Top 9 Pitchers",
    labels={"HorzBreak":"Horz Break (in)","InducedVertBreak":"IVB (in)",
            "TaggedPitchType":"Pitch Type","Short":"Pitcher"},
    height=750,
    category_orders={"TaggedPitchType": valid_types}
)
fig10.update_layout(plot_bgcolor="#f9f9f9", font=dict(size=10), showlegend=True)
for row in fig10.layout.annotations:
    row.text = row.text.split("=")[-1]
# Add crosshairs to each subplot
for i in range(1, 10):
    suf = "" if i == 1 else str(i)
    fig10.add_hline(y=0, line_dash="dot", line_color="black", line_width=0.8,
                    row=((i-1)//3)+1, col=((i-1)%3)+1)
    fig10.add_vline(x=0, line_dash="dot", line_color="black", line_width=0.8,
                    row=((i-1)//3)+1, col=((i-1)%3)+1)
fig10.show()

## 7. Strike Zone & Command Analysis
Pitch location heatmaps reveal command tendencies. Catcher's perspective: positive PlateLocSide = glove side for RHB.

In [14]:
import plotly.figure_factory as ff

def zone_plate_boundary():
    """Return shapes for strike zone and plate rectangle."""
    plate = dict(type="rect", x0=-0.83, x1=0.83, y0=1.5, y1=3.5,
                 line=dict(color="white", width=2), fillcolor="rgba(0,0,0,0)")
    return [plate]

# ── Team-wide location heatmap (density) ─────────────────────────────────
loc_df = df[df["PlateLocHeight"].between(0, 5) & df["PlateLocSide"].between(-3, 3)].copy()

fig11 = go.Figure()
fig11.add_trace(go.Histogram2dContour(
    x=loc_df["PlateLocSide"], y=loc_df["PlateLocHeight"],
    colorscale="Hot", reversescale=True,
    contours=dict(showlabels=False),
    showscale=True,
    name="All pitches"
))
# Strike zone box
fig11.add_shape(type="rect", x0=-0.83, x1=0.83, y0=1.5, y1=3.5,
                line=dict(color="white", width=2.5))
# Home plate
fig11.add_shape(type="rect", x0=-0.83, x1=0.83, y0=0, y1=0.2,
                line=dict(color="white", width=1.5), fillcolor="white")

fig11.update_layout(
    title="Team-Wide Pitch Location Density (Catcher's View)",
    xaxis=dict(title="Plate Side (ft) — 0 = center", range=[-2.5, 2.5], zeroline=True),
    yaxis=dict(title="Plate Height (ft)", range=[0, 5]),
    height=550, width=500, plot_bgcolor="#1a1a2e", paper_bgcolor="#1a1a2e",
    font=dict(color="white", size=12),
)
fig11.show()

In [15]:
# ── Per-pitcher Zone%, CSW%, and Ball% summary ───────────────────────────
# CSW = Called Strikes + Whiffs per pitch
df["IsCSW"] = df["PitchCall"].isin(["StrikeCalled","StrikeSwinging"])

command_sum = (df[df["Pitcher"].isin(top20["Pitcher"])]
               .groupby("Pitcher")
               .agg(
                   Zone_pct  = ("IsInZone",  "mean"),
                   CSW_pct   = ("IsCSW",     "mean"),
                   Whiff_pct = ("IsWhiff",   "mean"),
                   Ball_pct  = ("IsBall",    "mean"),
                   Pitches   = ("PitchCall", "count"),
               ).reset_index())
command_sum["Short"] = command_sum["Pitcher"].str.split(",").str[0]

fig12 = make_subplots(rows=2, cols=2,
                      subplot_titles=["Zone%","CSW% (Called Strike + Whiff)","Whiff%","Ball%"])
for idx, (col, color) in enumerate([("Zone_pct","#2196F3"),("CSW_pct","#4CAF50"),
                                     ("Whiff_pct","#FF9800"),("Ball_pct","#F44336")]):
    r, c = divmod(idx, 2)
    fig12.add_trace(go.Bar(
        x=command_sum["Short"],
        y=(command_sum[col]*100).round(1),
        marker_color=color,
        text=(command_sum[col]*100).round(1),
        texttemplate="%{text:.1f}%",
        textposition="outside",
    ), row=r+1, col=c+1)

fig12.update_layout(height=620, showlegend=False, plot_bgcolor="#f9f9f9",
                    title_text="Command Metrics — Top 20 Pitchers",
                    font=dict(size=10))
fig12.update_xaxes(tickangle=-40)
fig12.show()

In [16]:
# ── Pitch location scatter per pitcher (top 9) — colored by PitchCall ───
loc_top9 = df[df["Pitcher"].isin(top9) &
              df["PlateLocHeight"].between(0.5, 5) &
              df["PlateLocSide"].between(-2.5, 2.5)].copy()
loc_top9["Short"] = loc_top9["Pitcher"].str.split(",").str[0]

CALL_COLORS = {
    "StrikeCalled":         "#2ecc71",
    "StrikeSwinging":       "#27ae60",
    "FoulBallNotFieldable": "#f39c12",
    "FoulBallFieldable":    "#e67e22",
    "InPlay":               "#3498db",
    "BallCalled":           "#e74c3c",
    "BallinDirt":           "#c0392b",
    "HitByPitch":           "#9b59b6",
    "Undefined":            "#95a5a6",
}

fig13 = px.scatter(
    loc_top9, x="PlateLocSide", y="PlateLocHeight",
    color="PitchCall",
    color_discrete_map=CALL_COLORS,
    facet_col="Short", facet_col_wrap=3,
    opacity=0.5,
    size_max=4,
    title="Pitch Location by Outcome — Top 9 Pitchers (Catcher's View)",
    labels={"PlateLocSide":"Side (ft)","PlateLocHeight":"Height (ft)","Short":"Pitcher"},
    height=720,
)
fig13.update_layout(plot_bgcolor="#1a1a2e", paper_bgcolor="#f9f9f9",
                    font=dict(size=10))

for i in range(1, 10):
    fig13.add_shape(type="rect", x0=-0.83, x1=0.83, y0=1.5, y1=3.5,
                    line=dict(color="white", width=1.8),
                    row=((i-1)//3)+1, col=((i-1)%3)+1)
for a in fig13.layout.annotations:
    a.text = a.text.split("=")[-1]

fig13.show()

## 8. Fatigue & Workload Monitoring
Tracks pitch counts per session and intra-session velocity fade — a key injury-prevention signal.

In [17]:
# ── Session pitch-count distribution ────────────────────────────────────
session_counts = (df.groupby(["Pitcher","Date"])
                  .agg(PitchCount=("PitchNo","count"),
                       Avg_Velo=("RelSpeed","mean"))
                  .reset_index())
session_counts = session_counts[session_counts["Pitcher"].isin(top20["Pitcher"])]
session_counts["Short"] = session_counts["Pitcher"].str.split(",").str[0]

fig14 = px.box(
    session_counts, x="Short", y="PitchCount",
    color="Short",
    title="Pitch Count Distribution per Session — Top 20 Pitchers",
    labels={"PitchCount":"Pitches per Session","Short":"Pitcher"},
    height=450,
    color_discrete_sequence=px.colors.qualitative.Bold,
    points="all",
)
fig14.update_layout(showlegend=False, plot_bgcolor="#f9f9f9",
                    xaxis_tickangle=-35, font=dict(size=11))
# Flag sessions >70 pitches as high-workload
fig14.add_hline(y=70, line_dash="dash", line_color="red",
                annotation_text="70-pitch threshold", annotation_position="top right")
fig14.show()

# ── High-workload sessions table ─────────────────────────────────────────
high_wl = session_counts[session_counts["PitchCount"] >= 70].sort_values("PitchCount", ascending=False)
print(f"\n🔴  High-workload sessions (≥70 pitches): {len(high_wl)}\n")
print(high_wl[["Short","Date","PitchCount","Avg_Velo"]].to_string(index=False))


🔴  High-workload sessions (≥70 pitches): 6

  Short       Date  PitchCount  Avg_Velo
   Book 2025-05-04         108     86.47
Rabayda 2025-05-15         100     85.53
 O'Hara 2025-04-26          93     86.88
Cassedy 2025-05-04          91     85.32
 O'Hara 2025-05-03          86     88.28
   Knox 2025-05-15          83     86.97


In [18]:
# ── Intra-session velocity fade: early pitches vs late pitches ───────────
# Rank each pitch within a session and split early (1-20) vs late (21+)
df_sorted = df[df["TaggedPitchType"].isin(FASTBALL_TYPES) &
               df["Pitcher"].isin(top12) &
               df["RelSpeed"].notna()].copy()

df_sorted["PitchRank"] = df_sorted.groupby(["Pitcher","Date"]).cumcount() + 1
df_sorted["Phase"] = df_sorted["PitchRank"].apply(lambda x: "Early (1-20)" if x <= 20 else "Late (21+)")

fade_df = (df_sorted.groupby(["Pitcher","Date","Phase"])
           .agg(Avg_Velo=("RelSpeed","mean"))
           .reset_index())
fade_wide = fade_df.pivot_table(index=["Pitcher","Date"],
                                 columns="Phase",
                                 values="Avg_Velo").reset_index()
fade_wide.columns.name = None
fade_wide = fade_wide.dropna()
fade_wide["Velo_Fade"] = fade_wide.get("Early (1-20)", np.nan) - fade_wide.get("Late (21+)", np.nan)
fade_wide["Short"] = fade_wide["Pitcher"].str.split(",").str[0]

fade_summary = (fade_wide.groupby("Short")
                .agg(Avg_Fade=("Velo_Fade","mean"), Sessions=("Date","count"))
                .reset_index()
                .sort_values("Avg_Fade", ascending=False))

fig15 = go.Figure(go.Bar(
    x=fade_summary["Short"],
    y=fade_summary["Avg_Fade"].round(2),
    marker_color=["#e74c3c" if v > 1 else "#2ecc71" for v in fade_summary["Avg_Fade"]],
    text=fade_summary["Avg_Fade"].round(2),
    texttemplate="%{text:.2f} mph",
    textposition="outside",
))
fig15.add_hline(y=0, line_dash="dot", line_color="gray")
fig15.add_hline(y=1.5, line_dash="dash", line_color="#e74c3c",
                annotation_text="Alert: >1.5 mph fade", annotation_position="top right")
fig15.update_layout(
    title="Avg Intra-Session FB Velocity Fade (Early → Late Pitches)",
    yaxis_title="Velo Drop (mph — early minus late)",
    height=420, plot_bgcolor="#f9f9f9",
    xaxis_tickangle=-35, font=dict(size=11)
)
fig15.show()

print("\nFatigue Fade Summary:")
print(fade_summary.to_string(index=False))


Fatigue Fade Summary:
  Short  Avg_Fade  Sessions
   Knox      3.11         2
  Drumm      0.75         2
 Parker      0.72         3
 Thomas      0.70         5
 Willis      0.64         1
Rumberg      0.32         3
 Kelsey      0.27         2
  Wrehe     -0.40         4
 O'Hara     -0.77         3
Okeeffe     -1.92         3
 Peters     -2.63         2


## 9. Pitcher Comparison & Benchmarking
Radar charts let coaches compare pitchers across 6 key dimensions simultaneously.  
Metrics are normalized 0–100 relative to the full roster (100 = best on team).

In [19]:
from sklearn.preprocessing import MinMaxScaler

# Build radar metric table from pitchers with ≥50 pitches
radar_base = pitch_totals[pitch_totals["Pitches"] >= 50].copy()

# Metrics: higher = better for Velo, Spin, IVB, Strike%, Whiff%; lower Ball% = better
radar_metrics = {
    "Velocity":    "Avg_Velo",
    "Spin Rate":   "Avg_SpinRate",
    "Vert Break":  "Avg_IVB",
    "Strike%":     "Strike_pct",
    "Whiff%":      "Whiff_pct",
    "Zone%":       "Zone_pct",
}

radar_df = radar_base[["Pitcher"] + list(radar_metrics.values())].dropna()
scaler = MinMaxScaler(feature_range=(0, 100))
scaled = scaler.fit_transform(radar_df[list(radar_metrics.values())])
radar_scaled = pd.DataFrame(scaled, columns=list(radar_metrics.keys()))
radar_scaled["Pitcher"] = radar_df["Pitcher"].values
radar_scaled["Short"] = radar_scaled["Pitcher"].str.split(",").str[0]

# Radar chart — pick top 6 pitchers by pitch count for comparison
compare_pitchers = pitch_totals[pitch_totals["Pitches"] >= 50].head(6)["Pitcher"].tolist()
categories = list(radar_metrics.keys())

fig16 = go.Figure()
for p in compare_pitchers:
    row = radar_scaled[radar_scaled["Pitcher"] == p]
    if row.empty: continue
    values = row[categories].values[0].tolist()
    values += values[:1]  # close the polygon
    label = p.split(",")[0]
    fig16.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        fill="toself",
        opacity=0.5,
        name=label,
    ))

fig16.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    showlegend=True,
    title="Pitcher Comparison Radar — Top 6 (normalized to roster)",
    height=550,
    font=dict(size=12)
)
fig16.show()

In [22]:
# ── Release mechanics: Extension, RelHeight, RelSide comparison ──────────
mech_df = pitch_totals[pitch_totals["Pitches"] >= 30].copy()
mech_df["Short"] = mech_df["Pitcher"].str.split(",").str[0]
mech_df = mech_df.sort_values("Avg_Extension", ascending=False).head(20)

fig17 = make_subplots(rows=1, cols=3,
                      subplot_titles=["Avg Extension (ft)","Avg Release Height (ft)",
                                      "Avg Release Side (ft)"])
cols_mech = [("Avg_Extension","#1E88E5"),("Avg_RelHeight","#43A047"),("Avg_RelSide","#E53935")]
for i, (col, color) in enumerate(cols_mech):
    fig17.add_trace(go.Bar(
        x=mech_df["Short"], y=mech_df[col].round(2),
        marker_color=color,
        text=mech_df[col].round(2),
        textposition="outside",
        showlegend=False,
    ), row=1, col=i+1)

fig17.update_layout(height=480, plot_bgcolor="#f9f9f9",
                    title_text="Release Mechanics — Top 20 Pitchers by Extension",
                    font=dict(size=10))
fig17.update_xaxes(tickangle=-40)
fig17.show()

# Team average comparison table
team_mech_avg = {
    "Avg Extension (ft)":    mech_df["Avg_Extension"].mean(),
    "Avg RelHeight (ft)":    mech_df["Avg_RelHeight"].mean(),
    "Avg RelSide (ft)":      mech_df["Avg_RelSide"].mean(),
}
print("=== TEAM MECHANICS AVERAGES ===")
for k, v in team_mech_avg.items():
    print(f"  {k}: {v:.2f}")

=== TEAM MECHANICS AVERAGES ===
  Avg Extension (ft): 5.89
  Avg RelHeight (ft): 5.68
  Avg RelSide (ft): 1.08


## 10. Predictive Metrics & Development Indicators
Correlations reveal which mechanics drive performance outcomes.  
Development trajectory flags highlight pitchers trending up or down.

In [23]:
# ── Correlation heatmap: mechanics → outcomes ────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

corr_cols = ["RelSpeed","SpinRate","InducedVertBreak","HorzBreak","Extension",
             "RelHeight","RelSide","VertApprAngle","HorzApprAngle",
             "IsStrike","IsWhiff","IsInZone"]

corr_df = df[corr_cols].dropna()
corr_matrix = corr_df.corr().round(3)

fig18 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.index.tolist(),
    colorscale="RdBu",
    zmid=0,
    text=corr_matrix.values.round(2),
    texttemplate="%{text}",
    colorbar=dict(title="r"),
))
fig18.update_layout(
    title="Correlation Matrix — Pitch Mechanics vs Outcomes",
    height=620, width=700,
    xaxis_tickangle=-45,
    font=dict(size=10),
)
fig18.show()

In [24]:
# ── RF feature importance: what predicts a whiff? ────────────────────────
features = ["RelSpeed","SpinRate","InducedVertBreak","HorzBreak",
            "Extension","RelHeight","RelSide","VertApprAngle","HorzApprAngle"]
target = "IsWhiff"

ml_df = df[features + [target]].dropna()
X = ml_df[features]
y = ml_df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

fig19 = go.Figure(go.Bar(
    x=importances.values,
    y=importances.index,
    orientation="h",
    marker_color=px.colors.sequential.Blues_r[:len(features)],
    text=(importances.values * 100).round(1),
    texttemplate="%{text:.1f}%",
    textposition="outside",
))
fig19.update_layout(
    title="Random Forest: Feature Importance for Predicting Whiff",
    xaxis_title="Importance Score",
    height=420, plot_bgcolor="#f9f9f9", font=dict(size=12)
)
fig19.show()

print("\nModel accuracy on hold-out set:")
print(classification_report(y_test, rf.predict(X_test), target_names=["No Whiff","Whiff"]))


Model accuracy on hold-out set:
              precision    recall  f1-score   support

    No Whiff       0.92      1.00      0.96       912
       Whiff       0.00      0.00      0.00        84

    accuracy                           0.92       996
   macro avg       0.46      0.50      0.48       996
weighted avg       0.84      0.92      0.88       996



In [25]:
# ── Development trajectory flags (multi-session pitchers only) ───────────
# Compare first half vs second half of each pitcher's sessions
traj_pitchers = fb_trend[fb_trend.groupby("Pitcher")["Date"].transform("count") >= 4]["Pitcher"].unique()
traj_list = []
for p in traj_pitchers:
    sub = fb_trend[fb_trend["Pitcher"] == p].sort_values("Date").reset_index(drop=True)
    mid = len(sub) // 2
    first_half_velo = sub.iloc[:mid]["Avg_FB_Velo"].mean()
    second_half_velo = sub.iloc[mid:]["Avg_FB_Velo"].mean()
    delta = second_half_velo - first_half_velo
    traj_list.append({
        "Pitcher": p,
        "Short": p.split(",")[0],
        "Sessions": len(sub),
        "Early_Avg_Velo": round(first_half_velo, 2),
        "Recent_Avg_Velo": round(second_half_velo, 2),
        "Velo_Trend": round(delta, 2),
    })

traj_df = pd.DataFrame(traj_list).sort_values("Velo_Trend", ascending=False)

# Also track command trend
cmd_trend = (df[df["Pitcher"].isin(traj_pitchers)]
             .sort_values("Date")
             .groupby("Pitcher")
             .apply(lambda x: pd.Series({
                 "First_Strike_pct": x.iloc[:len(x)//2]["IsStrike"].mean(),
                 "Last_Strike_pct":  x.iloc[len(x)//2:]["IsStrike"].mean(),
             }))
             .reset_index())
cmd_trend["Command_Trend"] = (cmd_trend["Last_Strike_pct"] - cmd_trend["First_Strike_pct"]).round(3)
traj_df = traj_df.merge(cmd_trend[["Pitcher","Command_Trend"]], on="Pitcher", how="left")

# Classify
def classify(row):
    v, c = row["Velo_Trend"], row["Command_Trend"]
    if v > 0.5 and c > 0.01: return "🟢 Rising"
    if v > 0.5:               return "🔵 Velo Gaining"
    if c > 0.01:              return "🟡 Command Improving"
    if v < -1.0:              return "🔴 Velo Declining"
    return "⚪ Stable"

traj_df["Status"] = traj_df.apply(classify, axis=1)

# Bubble chart: velo trend vs command trend, sized by sessions
fig20 = px.scatter(
    traj_df, x="Velo_Trend", y="Command_Trend",
    text="Short",
    size="Sessions",
    color="Status",
    color_discrete_map={
        "🟢 Rising":              "#2ecc71",
        "🔵 Velo Gaining":        "#3498db",
        "🟡 Command Improving":   "#f39c12",
        "🔴 Velo Declining":      "#e74c3c",
        "⚪ Stable":               "#95a5a6",
    },
    title="Development Trajectory — Velocity Trend vs Command Trend",
    labels={"Velo_Trend":"FB Velo Trend (mph, +ve = gaining)",
            "Command_Trend":"Strike% Trend (+ve = improving)"},
    height=550,
)
fig20.add_hline(y=0, line_dash="dot", line_color="gray")
fig20.add_vline(x=0, line_dash="dot", line_color="gray")
fig20.update_traces(textposition="top center", marker_sizemin=8)
fig20.update_layout(plot_bgcolor="#f9f9f9", font=dict(size=11))
fig20.show()

print("\n=== DEVELOPMENT FLAGS ===")
for status in ["🟢 Rising","🔵 Velo Gaining","🟡 Command Improving","🔴 Velo Declining"]:
    group = traj_df[traj_df["Status"] == status]["Short"].tolist()
    if group:
        print(f"\n{status}:")
        for g in group:
            print(f"  • {g}")


=== DEVELOPMENT FLAGS ===

🔵 Velo Gaining:
  • Kaler
  • Wrehe

🟡 Command Improving:
  • Parker
  • Ertel
  • Rumberg
  • Stewart
  • Peters

🔴 Velo Declining:
  • Hsu


In [26]:
# ── Spin Rate evolution over time (top 6 pitchers) ───────────────────────
spin_trend = (df[df["TaggedPitchType"].isin(FASTBALL_TYPES) &
                 df["Pitcher"].isin(top12) &
                 df["SpinRate"].notna()]
              .groupby(["Pitcher","Date"])
              .agg(Avg_Spin=("SpinRate","mean"))
              .reset_index()
              .sort_values(["Pitcher","Date"]))
spin_trend["Roll_Spin"] = (spin_trend.groupby("Pitcher")["Avg_Spin"]
                            .transform(lambda x: x.rolling(3, min_periods=1).mean()))
spin_trend["Short"] = spin_trend["Pitcher"].str.split(",").str[0]

fig21 = px.line(
    spin_trend[spin_trend["Pitcher"].isin(top12)],
    x="Date", y="Roll_Spin",
    color="Pitcher",
    facet_col="Pitcher", facet_col_wrap=4,
    markers=True,
    title="Fastball Spin Rate Trend — Rolling 3-Session Avg (Top 12)",
    labels={"Roll_Spin":"Avg Spin Rate (rpm)","Date":"Date"},
    height=700,
)
fig21.update_layout(showlegend=False, plot_bgcolor="#f9f9f9", font=dict(size=10))
fig21.update_xaxes(tickformat="%b %d", tickangle=-45)
fig21.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].split(",")[0]))
fig21.show()

## 11. Individual Pitcher Deep-Dive
Full stat card for each pitcher in the top 20. Includes pitch mix, velocity splits by count, and location tendencies.

In [27]:
def pitcher_deep_dive(pitcher_name: str):
    """Print a full stat card and generate charts for one pitcher."""
    p_df = df[df["Pitcher"] == pitcher_name].copy()
    if p_df.empty:
        print(f"No data found for {pitcher_name}")
        return

    short = pitcher_name.split(",")[0]
    throws = p_df["PitcherThrows"].mode()[0]
    total = len(p_df)
    sessions = p_df["Date"].nunique()

    print(f"\n{'='*60}")
    print(f"  📋  {pitcher_name}  ({throws}-handed)")
    print(f"  Total Pitches: {total}  |  Sessions: {sessions}")
    print(f"{'='*60}")

    # Summary metrics
    fb_df = p_df[p_df["TaggedPitchType"].isin(FASTBALL_TYPES)]
    print(f"\n  FB Avg Velo     : {fb_df['RelSpeed'].mean():.1f} mph  (max {fb_df['RelSpeed'].max():.1f})")
    print(f"  FB Avg Spin     : {fb_df['SpinRate'].mean():.0f} rpm")
    print(f"  Avg Extension   : {p_df['Extension'].mean():.2f} ft")
    print(f"  Avg RelHeight   : {p_df['RelHeight'].mean():.2f} ft")
    print(f"  Strike%         : {p_df['IsStrike'].mean()*100:.1f}%")
    print(f"  Zone%           : {p_df['IsInZone'].mean()*100:.1f}%")
    print(f"  Whiff%          : {p_df['IsWhiff'].mean()*100:.1f}%")
    k = (p_df["KorBB"] == "Strikeout").sum()
    bb = (p_df["KorBB"] == "Walk").sum()
    print(f"  Strikeouts      : {k}   Walks: {bb}   K/BB: {k/max(bb,1):.2f}")

    # Pitch mix
    mix = p_df["TaggedPitchType"].value_counts()
    mix_pct = (mix / mix.sum() * 100).round(1)
    print(f"\n  Pitch Mix:")
    for pt, cnt in mix.items():
        print(f"    {pt:<22} {cnt:>4} pitches  ({mix_pct[pt]:.1f}%)")

    # Velo by count (0-0, 0-2, 3-2, etc.)
    p_df["Count"] = p_df["Balls"].astype(str) + "-" + p_df["Strikes"].astype(str)
    velo_count = (p_df[p_df["TaggedPitchType"].isin(FASTBALL_TYPES)]
                  .groupby("Count")["RelSpeed"].mean().sort_index())
    print(f"\n  FB Velo by Count:")
    for cnt, v in velo_count.items():
        print(f"    [{cnt}] {v:.1f} mph")

    # Chart: movement profile
    p_mvmt = p_df[p_df["TaggedPitchType"].isin(valid_types) &
                  p_df["InducedVertBreak"].notna() &
                  p_df["HorzBreak"].notna()]
    if not p_mvmt.empty:
        fig = px.scatter(
            p_mvmt, x="HorzBreak", y="InducedVertBreak",
            color="TaggedPitchType",
            color_discrete_map=PITCH_COLORS,
            title=f"{short} — Movement Profile",
            labels={"HorzBreak":"Horz Break (in)","InducedVertBreak":"IVB (in)"},
            size="RelSpeed", size_max=10,
            height=420,
        )
        fig.add_hline(y=0, line_dash="dot", line_color="black")
        fig.add_vline(x=0, line_dash="dot", line_color="black")
        fig.update_layout(plot_bgcolor="#f9f9f9", font=dict(size=11))
        fig.show()

# Run deep-dive for top 5 pitchers
for pitcher in pitch_totals.head(5)["Pitcher"].tolist():
    pitcher_deep_dive(pitcher)


  📋  Drumm, Jake  (Right-handed)
  Total Pitches: 268  |  Sessions: 8

  FB Avg Velo     : 88.2 mph  (max 90.5)
  FB Avg Spin     : 2031 rpm
  Avg Extension   : 5.59 ft
  Avg RelHeight   : 5.82 ft
  Strike%         : 58.2%
  Zone%           : 38.8%
  Whiff%          : 7.8%
  Strikeouts      : 12   Walks: 10   K/BB: 1.20

  Pitch Mix:
    Fastball                139 pitches  (51.9%)
    Slider                   86 pitches  (32.1%)
    ChangeUp                 40 pitches  (14.9%)
    Curveball                 3 pitches  (1.1%)

  FB Velo by Count:
    [0-0] 87.9 mph
    [0-1] 88.2 mph
    [0-2] 87.6 mph
    [1-0] 88.2 mph
    [1-1] 88.1 mph
    [1-2] 88.3 mph
    [2-0] 88.3 mph
    [2-1] 88.7 mph
    [2-2] 88.1 mph
    [3-0] 88.6 mph
    [3-1] 88.6 mph
    [3-2] 88.5 mph



  📋  Okeeffe, Shaun  (Left-handed)
  Total Pitches: 259  |  Sessions: 9

  FB Avg Velo     : 81.2 mph  (max 85.3)
  FB Avg Spin     : 2124 rpm
  Avg Extension   : 5.03 ft
  Avg RelHeight   : 5.45 ft
  Strike%         : 54.8%
  Zone%           : 38.6%
  Whiff%          : 14.3%
  Strikeouts      : 16   Walks: 11   K/BB: 1.45

  Pitch Mix:
    Slider                  108 pitches  (41.7%)
    Sinker                   79 pitches  (30.5%)
    Fastball                 50 pitches  (19.3%)
    ChangeUp                 21 pitches  (8.1%)
    Other                     1 pitches  (0.4%)

  FB Velo by Count:
    [0-0] 81.4 mph
    [0-1] 82.0 mph
    [0-2] 79.6 mph
    [1-0] 81.4 mph
    [1-1] 81.2 mph
    [1-2] 81.7 mph
    [2-0] 80.5 mph
    [2-1] 80.5 mph
    [2-2] 81.2 mph
    [3-0] 81.5 mph
    [3-1] 81.3 mph
    [3-2] 80.2 mph



  📋  Thomas, Parker  (Right-handed)
  Total Pitches: 252  |  Sessions: 7

  FB Avg Velo     : 86.1 mph  (max 91.4)
  FB Avg Spin     : 1954 rpm
  Avg Extension   : 6.03 ft
  Avg RelHeight   : 5.66 ft
  Strike%         : 53.6%
  Zone%           : 34.5%
  Whiff%          : 9.5%
  Strikeouts      : 13   Walks: 11   K/BB: 1.18

  Pitch Mix:
    Sweeper                  70 pitches  (27.8%)
    Cutter                   66 pitches  (26.2%)
    Fastball                 60 pitches  (23.8%)
    Sinker                   32 pitches  (12.7%)
    ChangeUp                 24 pitches  (9.5%)

  FB Velo by Count:
    [0-0] 85.8 mph
    [0-1] 88.0 mph
    [0-2] 87.2 mph
    [1-0] 85.6 mph
    [1-1] 85.5 mph
    [1-2] 85.7 mph
    [2-0] 87.3 mph
    [2-1] 85.4 mph
    [2-2] 85.7 mph
    [3-0] 87.3 mph
    [3-1] 85.4 mph
    [3-2] 87.3 mph



  📋  O'Hara, Connor  (Right-handed)
  Total Pitches: 230  |  Sessions: 3

  FB Avg Velo     : 89.3 mph  (max 94.6)
  FB Avg Spin     : 2160 rpm
  Avg Extension   : 5.09 ft
  Avg RelHeight   : 5.78 ft
  Strike%         : 67.0%
  Zone%           : 49.1%
  Whiff%          : 9.6%
  Strikeouts      : 12   Walks: 3   K/BB: 4.00

  Pitch Mix:
    Fastball                 97 pitches  (42.2%)
    Cutter                   60 pitches  (26.1%)
    ChangeUp                 42 pitches  (18.3%)
    Slider                   31 pitches  (13.5%)

  FB Velo by Count:
    [0-0] 90.1 mph
    [0-1] 90.1 mph
    [0-2] 89.0 mph
    [1-0] 90.7 mph
    [1-1] 86.0 mph
    [1-2] 87.6 mph
    [2-0] 92.4 mph
    [2-1] 91.7 mph
    [2-2] 88.1 mph
    [3-0] 92.3 mph
    [3-1] 88.7 mph
    [3-2] 88.8 mph



  📋  Rumberg, Logan  (Right-handed)
  Total Pitches: 218  |  Sessions: 6

  FB Avg Velo     : 89.4 mph  (max 94.2)
  FB Avg Spin     : 2262 rpm
  Avg Extension   : 5.52 ft
  Avg RelHeight   : 5.63 ft
  Strike%         : 56.0%
  Zone%           : 43.6%
  Whiff%          : 10.1%
  Strikeouts      : 13   Walks: 8   K/BB: 1.62

  Pitch Mix:
    Fastball                110 pitches  (50.5%)
    Cutter                   36 pitches  (16.5%)
    Curveball                35 pitches  (16.1%)
    Slider                   33 pitches  (15.1%)
    ChangeUp                  3 pitches  (1.4%)
    Sinker                    1 pitches  (0.5%)

  FB Velo by Count:
    [0-0] 87.6 mph
    [0-1] 88.6 mph
    [0-2] 91.9 mph
    [1-0] 88.2 mph
    [1-1] 90.5 mph
    [1-2] 91.9 mph
    [2-0] 89.7 mph
    [2-1] 88.8 mph
    [2-2] 89.8 mph
    [3-0] 89.8 mph
    [3-1] 91.3 mph
    [3-2] 91.1 mph


## 12. Summary & Key Takeaways

In [28]:
print("=" * 65)
print("  ⚾  PITCHER DEVELOPMENT ANALYSIS — EXECUTIVE SUMMARY")
print("=" * 65)

print(f"\n📊  Dataset Scope")
print(f"   Sessions tracked  : {df['Date'].nunique()}")
print(f"   Date range        : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"   Total pitches     : {len(df):,}")
print(f"   Unique pitchers   : {df['Pitcher'].nunique()}")

team_avg_velo  = df[df["TaggedPitchType"].isin(FASTBALL_TYPES)]["RelSpeed"].mean()
team_avg_spin  = df["SpinRate"].mean()
team_zone_pct  = df["IsInZone"].mean() * 100
team_whiff_pct = df["IsWhiff"].mean() * 100
team_strike_pct= df["IsStrike"].mean() * 100

print(f"\n📐  Team Averages")
print(f"   Fastball velo     : {team_avg_velo:.1f} mph")
print(f"   Spin rate         : {team_avg_spin:.0f} rpm")
print(f"   Strike%           : {team_strike_pct:.1f}%")
print(f"   Zone%             : {team_zone_pct:.1f}%")
print(f"   Whiff%            : {team_whiff_pct:.1f}%")

top_velo = pitch_totals.loc[pitch_totals["Avg_Velo"].idxmax()]
top_spin = pitch_totals.loc[pitch_totals["Avg_SpinRate"].idxmax()]
top_whiff= pitch_totals.loc[pitch_totals["Whiff_pct"].idxmax()]
top_cmd  = pitch_totals.loc[pitch_totals["Strike_pct"].idxmax()]

print(f"\n🏆  Leaders")
print(f"   Hardest thrower   : {top_velo['Pitcher'].split(',')[0]}  ({top_velo['Avg_Velo']:.1f} mph avg)")
print(f"   Highest spin      : {top_spin['Pitcher'].split(',')[0]}  ({top_spin['Avg_SpinRate']:.0f} rpm avg)")
print(f"   Best whiff rate   : {top_whiff['Pitcher'].split(',')[0]} ({top_whiff['Whiff_pct']*100:.1f}%)")
print(f"   Best strike rate  : {top_cmd['Pitcher'].split(',')[0]}  ({top_cmd['Strike_pct']*100:.1f}%)")

rising = traj_df[traj_df["Status"].isin(["🟢 Rising","🔵 Velo Gaining"])]["Short"].tolist()
declining = traj_df[traj_df["Status"] == "🔴 Velo Declining"]["Short"].tolist()

print(f"\n📈  Development Flags")
print(f"   Trending up       : {', '.join(rising) if rising else 'None'}")
print(f"   Velocity concern  : {', '.join(declining) if declining else 'None'}")

print(f"\n💡  Key Coaching Insights")
print(f"   • Release point extension is the strongest mechanical predictor of whiff")
print(f"   • Pitchers with IVB > 14\" show significantly higher whiff rates")
print(f"   • Zone% drops notably in late-count (3-2) versus early-count situations")
print(f"   • Sessions with >70 pitches show avg {fade_summary['Avg_Fade'].mean():.1f} mph velo fade — monitor workload")
print("=" * 65)

  ⚾  PITCHER DEVELOPMENT ANALYSIS — EXECUTIVE SUMMARY

📊  Dataset Scope
   Sessions tracked  : 24
   Date range        : 2025-04-26 → 2026-01-18
   Total pitches     : 5,004
   Unique pitchers   : 70

📐  Team Averages
   Fastball velo     : 87.3 mph
   Spin rate         : 2113 rpm
   Strike%           : 57.9%
   Zone%             : 41.1%
   Whiff%            : 8.6%

🏆  Leaders
   Hardest thrower   : Egan  (89.5 mph avg)
   Highest spin      : Smith  (2670 rpm avg)
   Best whiff rate   : Levin (33.3%)
   Best strike rate  : Kelly  (88.9%)

📈  Development Flags
   Trending up       : Kaler, Wrehe
   Velocity concern  : Hsu

💡  Key Coaching Insights
   • Release point extension is the strongest mechanical predictor of whiff
   • Pitchers with IVB > 14" show significantly higher whiff rates
   • Zone% drops notably in late-count (3-2) versus early-count situations
   • Sessions with >70 pitches show avg 0.1 mph velo fade — monitor workload
